# SpaceX Falcon 9
# 2: Web Scraping from Wikipedia

## What is Web Scraping?
Web scraping means extracting data directly from a website's HTML code.
Instead of an API giving us clean JSON, we must:
1. Download the raw HTML page
2. Parse (read & understand) the HTML structure
3. Find the tables we want
4. Extract the data from those tables
5. Convert into a clean Pandas DataFrame

## Why do we need this?
Not all data has an API. Wikipedia has rich Falcon 9 launch
history in HTML tables, so we scrape it directly.

## Tool we use: BeautifulSoup 
BeautifulSoup is a Python library that reads HTML like a 
structured document, thus letting us search, navigate and extract
specific parts of a webpage easily.

## What is HTML?
HTML is the code that builds every webpage. Data in webpages
is often stored in tags like:
- `<table>` → a table
- `<tr>`    → a table row
- `<th>`    → a table header cell
- `<td>`    → a table data cell

## Installing required libraries
- `BeautifulSoup`  → parses HTML pages
- `requests`    → downloads the HTML page from the web

In [ ]:
!pip3 install beautifulsoup4
!pip3 install requests

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


## Step 1: Import Libraries

- `requests`     → to download the HTML page
- `BeautifulSoup`→ to parse and navigate the HTML
- `re`           → regular expressions (pattern matching in text)
- `unicodedata`  → handles special characters in text
- `pandas`       → to store scraped data in a DataFrame

In [2]:
import sys
import requests
from bs4 import BeautifulSoup   # The main scraping tool
import re                        # For pattern matching in text
import unicodedata               # For handling special characters
import pandas as pd

## Step 2: Understand the Helper Functions

Before scraping, 5 helper functions are provided to clean
the messy raw HTML text. Wikipedia tables contain:
- Extra spaces and newlines
- Reference links like [8] or [e]
- Special unicode characters
- Nested HTML tags inside cells

### The 5 helper functions:
| Function                  | What it does                              |
|---------------------------|-------------------------------------------|
| date_time(cell)           | Extracts date and time from a cell        |
| booster_version(cell)     | Extracts booster name e.g. "F9 B5"       |
| landing_status(cell)      | Extracts landing result e.g. "Success"   |
| get_mass(cell)            | Extracts payload mass in kg               |
| extract_column_from_header| Cleans column header names                |

In [3]:
def date_time(table_cells):
    """
    Extracts date and time from an HTML table cell.
    Returns a list: [date_string, time_string]
    e.g. ['4 June 2010', '18:45']
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]


def booster_version(table_cells):
    """
    Extracts the booster version name from an HTML table cell.
    Skips every other item to avoid picking up reference numbers.
    e.g. 'Falcon 9 v1.0 B0003'
    """
    out = ''.join([booster_version for i, booster_version in
                   enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out


def landing_status(table_cells):
    """
    Extracts the booster landing result from an HTML table cell.
    e.g. 'Success', 'Failure', 'No attempt'
    """
    out = [i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    """
    Extracts payload mass from an HTML table cell.
    Normalizes unicode, then cuts string at 'kg'.
    e.g. '525 kg'
    Returns 0 if no mass found.
    """
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass


def extract_column_from_header(row):
    """
    Cleans up an HTML table header cell to get a plain column name.
    Removes: hyperlinks <a>, line breaks <br>, footnotes <sup>
    e.g. turns a messy <th> tag into 'Launch site'
    """
    if (row.br):
        row.br.extract()       # remove line break tags
    if row.a:
        row.a.extract()        # remove hyperlink tags
    if row.sup:
        row.sup.extract()      # remove footnote tags

    colunm_name = ' '.join(row.contents)

    # Ignore cells that are just numbers (not real column names)
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name

print(" Helper functions defined successfully")

 Helper functions defined successfully


## Step 3: Request the Wikipedia Page

### What are we doing?
We download the raw HTML of the Wikipedia page using requests.get()
exactly like we did with the SpaceX API, but this time instead of
getting clean JSON back, we get a massive HTML string.

### What is a static URL?
Instead of scraping the LIVE Wikipedia page (which changes daily),
we use a snapshot from 9th June 2021.
This keeps our results consistent with the course's expected answers.

### What are headers?
When your browser visits a website it sends information about itself
called "headers". Some websites BLOCK requests that don't look like
a real browser, they think it's a bot.

We send a fake browser header (User-Agent) so Wikipedia
doesn't block our request:
- User-Agent: "I am Chrome browser on Windows" (even though we are Python)

### What is BeautifulSoup?
BeautifulSoup takes the raw HTML string and builds a navigable
tree structure from it, like a map of the page.

For example this raw HTML string:

    "<table><tr><td>hi</td></tr></table>"

Gets parsed into a navigable tree:

    table
      tr
        td
          "hi"

Now we can SEARCH and NAVIGATE the tree instead of reading raw text.

In [4]:
# We use a Wikipedia SNAPSHOT from June 9, 2021
# oldid=1027686922 pins it to that exact version, results won't change even if Wikipedia edits the page later

static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

# Headers make our request look like it comes from a real browser
# Without this, Wikipedia might return a different page or block us
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

print("URL and headers defined")
print("URL:", static_url[:60], "...")

URL and headers defined
URL: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_ ...


## Step 3B: Send GET Request & Parse HTML

### Two steps happening here:

Step 1 - requests.get(url, headers) downloads the raw HTML as a string

Step 2 - BeautifulSoup(html, 'html.parser') parses it into a navigable tree

### What is 'html.parser'?
It tells BeautifulSoup WHICH parser (reading engine) to use.

- 'html.parser' = Python's built-in HTML reader, no extra install needed
- Other options exist like lxml and html5lib
- html.parser is the standard safe choice for most tasks

### How to verify it worked?
We use soup.title to find the page title tag in the HTML.

A Wikipedia page title looks like this:
"List of Falcon 9 and Falcon Heavy launches - Wikipedia"

If we see that printed out, our soup object was built correctly
and the page downloaded successfully.

In [5]:
# Step 1: Send GET request to download the raw HTML page
# We pass headers= so Wikipedia doesn't block us
response = requests.get(static_url, headers=headers)

# Check it was successful
print("Status code:", response.status_code)  # Should be 200

# Step 2: Parse the raw HTML into a BeautifulSoup object
# response.text = the full HTML as a giant string
# 'html.parser' = the built-in Python HTML reading engine
soup = BeautifulSoup(response.text, 'html.parser')

print("BeautifulSoup object created successfully")

Status code: 200
BeautifulSoup object created successfully


In [6]:
# soup.title finds the <title> tag in the HTML
# .string extracts the text inside it
print("Page title:", soup.title.string)

# Let's also see how big the HTML is
print("HTML length:", len(response.text), "characters")

Page title: List of Falcon 9 and Falcon Heavy launches - Wikipedia
HTML length: 1834675 characters


## Step 3C: Find the Launch Tables

### How does Wikipedia store launch data?
The launch records are inside HTML `<table>` tags with
specific CSS class names:
"wikitable plainrowheaders collapsible"

### How do we find them?
soup.find_all('table', 'wikitable plainrowheaders collapsible')

This searches the ENTIRE HTML tree and returns EVERY table
that has that exact class, like using Ctrl+F on a webpage.

### Why multiple tables?
Wikipedia splits launches by year, one table per year.
So we get back a LIST of tables, one for each year.

### What does a table look like in HTML?
<table>
  <tr>              ← table row
    <th>Flight No.</th>   ← header cell
    <th>Date</th>         ← header cell
  </tr>
  <tr>
    <td>1</td>      ← data cell
    <td>4 June 2010</td>
  </tr>
</table>

In [7]:
# Find every launch table on the page
# All launch tables share this specific CSS class name
html_tables = soup.find_all('table',
              'wikitable plainrowheaders collapsible')

# How many year-tables did we find?
print("Number of launch tables found:", len(html_tables))

# Let's peek at the first few rows of the FIRST table
# to confirm it looks like launch data
first_table = html_tables[0]
print()
print("First table preview (first 500 characters of HTML):")
print(str(first_table)[:500])

Number of launch tables found: 9

First table preview (first 500 characters of HTML):
<table class="wikitable plainrowheaders collapsible" style="width: 100%;">
<tbody><tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11"><span class="cite-bracket">[<


## Step 3D: Extract Column Names from Table Headers

### Why do we need column names?
Before storing any data we need to know WHAT each column
represents. Column names come from the `<th>` tags in the
first `<tr>` (header row) of each table.

### The process:
1. Find the first table's header row
2. Loop through every `<th>` tag in that row
3. Apply extract_column_from_header() to clean each name
4. Store non-empty names in a list called column_names

### What column names do we expect?
From the Wikipedia table we should get names like:
Flight No. | Date and time | Version Booster |
Launch site | Payload | Payload mass | Orbit |
Customer | Launch outcome | Booster landing

In [8]:
# Get the header row of the first table
# find('tr') finds the FIRST <tr> tag = the header row
column_names = []

# Loop through every header cell <th> in the first table's first row
first_row = html_tables[0].find('tr')

for th in first_row.find_all('th'):
    # Apply our helper function to clean the header text
    name = extract_column_from_header(th)

    # Only add it if it's not empty and not None
    if name is not None and len(name) > 0:
        column_names.append(name)

# See what column names we extracted
print("Extracted column names:")
print(column_names)
print()
print("Total columns found:", len(column_names))

Extracted column names:
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']

Total columns found: 8


## Step 4: Create the Launch Dictionary

### What is a dictionary here?
Before we can store scraped data, we need an empty container
with the right column names as keys.

Think of it like setting up an empty spreadsheet with headers:

    Flight No. | Launch site | Payload | Payload mass | ...
    -----------|-------------|---------|--------------|----
    (empty)    | (empty)     | (empty) | (empty)      | ...

We will fill one row at a time as we loop through the HTML tables.

### Why do we delete 'Date and time ( )'?
That column name has a leftover clock icon artifact from Wikipedia.
We delete it and instead add clean separate 'Date' and 'Time' columns.

### Extra columns we add manually:
- 'Version Booster'  → booster name e.g. 'F9 B5'
- 'Booster landing'  → landing result e.g. 'Success'
- 'Date'             → clean date e.g. '4 June 2010'
- 'Time'             → clean time e.g. '18:45'

In [9]:
# Create a dictionary using our extracted column names as keys
# dict.fromkeys() creates a dict where every key starts as None
launch_dict = dict.fromkeys(column_names)

# 'Date and time ( )' is a messy artifact from Wikipedia's clock icon
# We remove it and replace with clean 'Date' and 'Time' columns below
del launch_dict['Date and time ( )']

# Initialize every key with an empty list
# Each list will grow by one item per launch row we scrape
launch_dict['Flight No.']      = []
launch_dict['Launch site']     = []
launch_dict['Payload']         = []
launch_dict['Payload mass']    = []
launch_dict['Orbit']           = []
launch_dict['Customer']        = []
launch_dict['Launch outcome']  = []

# These columns weren't in the header so we add them manually
launch_dict['Version Booster'] = []   # booster name from row data
launch_dict['Booster landing'] = []   # landing result from row data
launch_dict['Date']            = []   # clean date split from datetime
launch_dict['Time']            = []   # clean time split from datetime

# Confirm the dictionary is set up correctly
print("Dictionary keys (column names):")
for key in launch_dict.keys():
    print(" ", key)

Dictionary keys (column names):
  Flight No.
  Launch site
  Payload
  Payload mass
  Orbit
  Customer
  Launch outcome
  Version Booster
  Booster landing
  Date
  Time
